# State data: audit and preparation

**Purpose:** check the existing SSA state files and create an analysis table for EDA. One row is a **state × recorded sex × birth year × name** count. The original files stay unchanged.

**Inputs:** 51 files in `data/raw/state/`; the SHA-256 inventory in `docs/data_inventory/files.csv` identifies the snapshot. Original acquisition date/release is unknown. National data and Census regional enrichment are outside this notebook.

**Run all cells in order** with the project environment. Preparation stops if structural checks fail. Unusual names are reviewed separately, without automatic deletion.

**Executed and verified (14 November 2026):** all code cells passed in the project `.venv` (Python 3.12.0, pandas 2.3.3). The 6,696,687-row export was reloaded and checked; raw hashes and original values are unchanged. Original acquisition provenance remains unresolved.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
from IPython.display import display

# Locate the project from the repository, notebook folder, or parent course folder.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'group_project_dsfm']
ROOT = next((p for p in candidates if (p / 'src/ssa_inventory.py').is_file()), None)
assert ROOT is not None, 'Open the repository or its notebooks folder.'
# Reuse the state-code list already defined by the inventory script.
sys.path.insert(0, str(ROOT))
from src.ssa_inventory import STATES

RAW = ROOT / 'data/raw/state'
OUT = ROOT / 'data/processed'
YEAR_START, YEAR_END = 1910, 2025
# These four fields identify one published record; count is its frequency.
KEY = ['state', 'sex', 'year', 'name']
COLUMNS = ['state', 'sex', 'year', 'name', 'count']
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 12)
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__}')
print(f'Interpreter: {sys.executable}')

Python 3.12.0 | pandas 2.3.3
Interpreter: /Users/jayyy/Library/Mobile Documents/3L68KQB4HG~com~readdle~CommonDocuments/Documents/HEC/Semester 1/Data science Methodes for MANA/group_project_dsfm/.venv/bin/python


## 1. Data dictionary

| Field | Type | Meaning |
|---|---|---|
| `state` | Category | Two-letter state of birth, including DC |
| `sex` | Category | Recorded sex (`F` or `M`), not an inferred identity |
| `year` | Integer / temporal | Birth year; expected 1910–2025 for this snapshot |
| `name` | Text | Published spelling; variants remain distinct |
| `count` | Integer | Published number of applications for the combination |

Key: `(state, sex, year, name)`. A missing row may reflect suppression; it is not automatically zero. Published state counts are neither a complete census of births nor a replacement for the national dataset. SSA retains entries such as `Unknown` and `Baby`, and may contain recording errors. Sources: [SSA qualifications](https://www.ssa.gov/oact/babynames/background.html), [publication limits](https://www.ssa.gov/oact/babynames/limits.html).

**Acceptance:** unchanged inputs; unique keys; no blank fields; valid codes and integer values; counts ≥5; and all 51 × 2 × 116 = **11,832 state–sex–year groups** present. Unusual spellings are review flags, not automatic errors.

## 2. Verify and load the local snapshot

Files must match the earlier SHA-256 inventory. Explicit column types catch malformed numeric values. `keep_default_na=False` prevents automatic conversion of text tokens to missing values; blank fields are checked separately.

In [2]:
# Match the available files to the recorded snapshot before loading data.
manifest = pd.read_csv(ROOT / 'docs/data_inventory/files.csv')
manifest = manifest.loc[manifest['dataset'].eq('state')].copy()
paths = sorted(RAW.glob('*.TXT'))
assert len(paths) == len(manifest) == 51
assert {p.stem for p in paths} == STATES
assert {p.relative_to(ROOT).as_posix() for p in paths} == set(manifest['relative_path'])

def sha256(path):
    """Return the fingerprint of a file's exact bytes without changing it."""
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

DTYPES = {'state': 'string', 'sex': 'string', 'year': 'int32',
          'name': 'string', 'count': 'int64'}
frames = []
for item in manifest.itertuples(index=False):
    path = ROOT / item.relative_path
    # A different fingerprint means the source no longer matches the inventory.
    assert sha256(path) == item.sha256, f'Input changed: {path.name}'
    # Preserve literal text values; blank fields are audited separately.
    frame = pd.read_csv(path, header=None, names=COLUMNS, dtype=DTYPES,
                        keep_default_na=False)
    assert len(frame) == item.row_count, f'Row-count mismatch: {path.name}'
    assert frame['state'].eq(path.stem).all(), f'State/filename mismatch: {path.name}'
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
del frames, frame
# Categories reduce memory for repeated codes without changing their labels.
for column in ['state', 'sex']:
    data[column] = data[column].astype('category')
assert len(data) == int(manifest['row_count'].sum())
print(f'{len(data):,} records | {data["state"].nunique()} states/DC | '
      f'{data["year"].min()}–{data["year"].max()}')
display(data.head(3))

6,696,687 records | 51 states/DC | 1910–2025


,state,sex,year,name,count
0,AK,F,1910,Mary,14
1,AK,F,1910,Annie,12
2,AK,F,1910,Anna,10


## 3. Structural checks and coverage

Duplicate keys are reported, not removed. Count values below five or years outside the verified period stop preparation. Coverage concerns observed **state–sex–year groups**; it does not imply that every possible name is published.

In [3]:
GROUP = ['state', 'sex', 'year']
# Empty strings are distinct from nulls because automatic NA parsing is disabled.
blank = {
    column: int(data[column].astype('string').str.fullmatch(r'\s*', na=False).sum())
    for column in ['state', 'sex', 'name']
}
# Count surplus duplicates separately from all rows involved in duplicate keys.
duplicate_excess = int(data.duplicated(KEY).sum())
duplicate_rows = int(data.duplicated(KEY, keep=False).sum())
checks = pd.Series({
    'Missing field values': int(data.isna().sum().sum()),
    'Blank text fields': sum(blank.values()),
    'Invalid state codes': int((~data['state'].isin(STATES)).sum()),
    'Invalid recorded-sex codes': int((~data['sex'].isin(['F', 'M'])).sum()),
    'Years outside 1910–2025': int((~data['year'].between(YEAR_START, YEAR_END)).sum()),
    'Counts below 5': int(data['count'].lt(5).sum()),
    'Excess duplicate-key rows': duplicate_excess,
}, name='Failures')

# Compare observed groups with the complete state–sex–year grid.
# This does not create zero-count rows for unreported names.
coverage = data.groupby(GROUP, observed=True, sort=True).agg(
    records=('name', 'size'), published_count=('count', 'sum')
)
expected = pd.MultiIndex.from_product(
    [sorted(STATES), ['F', 'M'], range(YEAR_START, YEAR_END + 1)], names=GROUP
)
missing_groups = expected.difference(coverage.index)
unexpected_groups = coverage.index.difference(expected)
checks.loc['Missing state–sex–year groups'] = len(missing_groups)
checks.loc['Unexpected state–sex–year groups'] = len(unexpected_groups)
display(checks.to_frame().assign(Passed=checks.eq(0)))
print(f'Observed groups: {len(coverage):,}/{len(expected):,}; '
      f'rows belonging to duplicate keys: {duplicate_rows:,}.')
if len(missing_groups):
    display(missing_groups.to_frame(index=False).head(10))
assert checks.eq(0).all(), 'Resolve failed checks before preparation.'

,Failures,Passed
Missing field values,0,True
Blank text fields,0,True
Invalid state codes,0,True
Invalid recorded-sex codes,0,True
Years outside 1910–2025,0,True
Counts below 5,0,True
Excess duplicate-key rows,0,True
Missing state–sex–year groups,0,True
Unexpected state–sex–year groups,0,True


Observed groups: 11,832/11,832; rows belonging to duplicate keys: 0.


## 4. Review unusual entries

Review flags cover whitespace, short/nonalphabetic spellings and a small explicit list of placeholder-like labels. They identify sensitivity checks, not proven errors. Large counts may simply reflect popular names; inspect their context rather than remove them by an arbitrary outlier rule.

In [4]:
names = data['name']
# These labels trigger review, not automatic removal.
placeholder_labels = {'baby', 'babyboy', 'babygirl', 'unknown', 'unnamed', 'infant'}
review_masks = {
    'Surrounding whitespace': names.ne(names.str.strip()),
    'Fewer than two characters': names.str.len().lt(2),
    'Nonalphabetic spelling': ~names.str.fullmatch(r'[A-Za-z]+', na=False),
    'Placeholder-like label': names.str.lower().isin(placeholder_labels),
}
review_counts = {label: int(mask.sum()) for label, mask in review_masks.items()}
display(pd.Series(review_counts, name='Flagged records').to_frame())
# A record appears in the review selection once, even if several flags apply.
flagged = pd.Series(False, index=data.index)
for mask in review_masks.values():
    flagged |= mask
if flagged.any():
    display(data.loc[flagged].groupby('name', observed=True).agg(
        records=('count', 'size'), published_count=('count', 'sum')
    ).sort_values('published_count', ascending=False).head(10))
# Inspect extreme counts in context: popularity alone is not a data error.
display(data['count'].describe(percentiles=[0.5, 0.95, 0.99]).to_frame())
display(data.nlargest(5, 'count')[COLUMNS])
print('No records have been removed or corrected.')

,Flagged records
Surrounding whitespace,0
Fewer than two characters,0
Nonalphabetic spelling,0
Placeholder-like label,1811


,records,published_count
name,,
Unknown,785,15046
Baby,609,8955
Infant,134,7761
Babyboy,139,1323
Babygirl,123,1166
Unnamed,21,484


,count
count,6.696687e+06
mean,4.942425e+01
std,1.689818e+02
min,5.000000e+00
50%,1.200000e+01
95%,1.860000e+02
99%,6.530000e+02
max,1.002700e+04


,state,sex,year,name,count
4460953,NY,M,1947,Robert,10027
4460954,NY,M,1947,John,9639
4460248,NY,M,1946,Robert,9294
4475394,NY,M,1963,Michael,9236
4470264,NY,M,1958,Michael,9226


No records have been removed or corrected.


## 5. Prepare an EDA table

Retain all published records and original spellings. Add four variables with specific uses:

| Variable | Definition and purpose |
|---|---|
| `decade` | Start year of the decade; temporal grouping |
| `name_length` | Number of characters; spelling descriptor |
| `published_share` | `count / sum(count)` within state × sex × year; share of published counts, **not all births** |
| `rank` | Count descending, then name alphabetically for ties; matches SSA's ranking convention |

No additional count threshold or imputation is applied. Female share is deferred until suppressed/missing opposite-sex records have an explicit treatment. The nine-column output supports the research questions; derived variables are not independent new information.

In [5]:
# Sort counts descending and names alphabetically to define rank ties.
prepared = data.sort_values(
    GROUP + ['count', 'name'], ascending=[True, True, True, False, True]
).copy()
prepared['decade'] = (prepared['year'] // 10 * 10).astype('int16')
prepared['name_length'] = prepared['name'].str.len().astype('int16')
groups = prepared.groupby(GROUP, observed=True, sort=False)
# The denominator includes only published counts in the same state–sex–year group.
prepared['published_share'] = prepared['count'] / groups['count'].transform('sum')
prepared['rank'] = (groups.cumcount() + 1).astype('int32')

# Preserve every original value and row; only order and new columns change.
pd.testing.assert_frame_equal(prepared[COLUMNS].sort_index(), data)
# Group shares must sum to one, allowing only floating-point rounding.
share_sums = prepared.groupby(GROUP, observed=True)['published_share'].sum()
assert share_sums.sub(1).abs().lt(1e-12).all()
assert prepared['published_share'].between(0, 1).all()
assert prepared.groupby(GROUP, observed=True)['rank'].max().eq(coverage['records']).all()
display(pd.DataFrame({
    'Before': [len(data), data['count'].sum(), data.duplicated(KEY).sum()],
    'After': [len(prepared), prepared['count'].sum(), prepared.duplicated(KEY).sum()]
}, index=['Records', 'Published count total', 'Excess duplicate-key rows']))
display(prepared.head(5))

,Before,After
Records,6696687,6696687
Published count total,330978761,330978761
Excess duplicate-key rows,0,0


,state,sex,year,name,count,decade,name_length,published_share,rank
0,AK,F,1910,Mary,14,1910,4,0.205882,1
1,AK,F,1910,Annie,12,1910,5,0.176471,2
2,AK,F,1910,Anna,10,1910,4,0.147059,3
3,AK,F,1910,Margaret,8,1910,8,0.117647,4
4,AK,F,1910,Helen,7,1910,5,0.102941,5


## 6. Save and verify

Save a compressed CSV and a small audit record in the ignored `data/processed/` folder. Reload the CSV in chunks, compare all non-share columns exactly, and allow only floating-point rounding for shares. Verify raw hashes again before replacing the output. The JSON records actual checks and output identity, not proof of the original download provenance.

In [6]:
# Verify a temporary export before replacing the final prepared file.
OUT.mkdir(exist_ok=True)
destination = OUT / 'state_names.csv.gz'
temporary = OUT / 'state_names.csv.gz.tmp'
# A fixed gzip timestamp avoids time-dependent differences between reruns.
prepared.to_csv(
    temporary, index=False, lineterminator='\n',
    compression={'method': 'gzip', 'compresslevel': 1, 'mtime': 0}
)

# Reload in chunks to limit memory; only shares allow a rounding tolerance.
offset = 0
reloaded_total = 0
exact_columns = [c for c in prepared.columns if c != 'published_share']
for chunk in pd.read_csv(
    temporary, compression='gzip', dtype=prepared.dtypes.to_dict(),
    keep_default_na=False, float_precision='round_trip', chunksize=250_000
):
    reference = prepared.iloc[offset:offset + len(chunk)].reset_index(drop=True)
    chunk = chunk.reset_index(drop=True)
    pd.testing.assert_frame_equal(chunk[exact_columns], reference[exact_columns])
    assert chunk['published_share'].sub(reference['published_share']).abs().lt(1e-12).all()
    offset += len(chunk)
    reloaded_total += int(chunk['count'].sum())
assert offset == len(prepared)
assert reloaded_total == int(data['count'].sum())
# Confirm that processing left the source files unchanged.
for item in manifest.itertuples(index=False):
    assert sha256(ROOT / item.relative_path) == item.sha256, 'Raw input changed during preparation.'
temporary.replace(destination)

# Record checks and file identity, without claiming original download provenance.
audit_record = {
    'input_inventory_sha256': sha256(ROOT / 'docs/data_inventory/files.csv'),
    'records': len(prepared),
    'published_count_total': reloaded_total,
    'observed_years': [int(data['year'].min()), int(data['year'].max())],
    'state_sex_year_groups': len(coverage),
    'structural_failures': {name: int(value) for name, value in checks.items()},
    'review_flags': review_counts,
    'rows_dropped': 0,
    'raw_fields_changed': 0,
    'raw_hashes_unchanged': True,
    'export_reload_verified': True,
    'output': destination.name,
    'output_sha256': sha256(destination),
    'python_version': sys.version.split()[0],
    'pandas_version': pd.__version__,
    'remaining': ['Original acquisition provenance', 'National dataset', 'Census regional lookup']
}
(OUT / 'state_audit.json').write_text(
    json.dumps(audit_record, indent=2) + '\n', encoding='utf-8'
)
print(f'Saved and reloaded {offset:,} records: {destination.relative_to(ROOT)}')
print('Raw files unchanged; all original values retained.')
print('Next: focused state-level EDA. Acquisition provenance remains unresolved.')

Saved and reloaded 6,696,687 records: data/processed/state_names.csv.gz
Raw files unchanged; all original values retained.
Next: focused state-level EDA. Acquisition provenance remains unresolved.


## 7. Report the verified preparation

This cell runs only after the export has passed its reload check. It writes the factual preparation summary included by the Quarto report, so the report cannot silently rely on unexecuted audit code.

In [7]:
# Build the report text from the audit that was actually completed above.
assert audit_record['export_reload_verified']
assert audit_record['output_sha256'] == sha256(destination)
report_lines = [
    f"The preparation retained **{len(prepared):,} records** across "
    f"**{data['state'].nunique()} states/DC** and **{YEAR_START}–{YEAR_END}**. "
    f"The rows contain **{data['name'].nunique():,} distinct spellings** and "
    f"sum to **{reloaded_total:,} published counts** across the period. "
    "These aggregate records are not individual people or independent observations.",
    "",
    "| Audit check | Result |",
    "|---|---:|",
]
for label, failures in checks.items():
    report_lines.append(f"| {label} | {int(failures):,} |")
report_lines += [
    f"| Observed state–sex–year groups | {len(coverage):,} |",
    "| Raw-file SHA-256 checks unchanged | 51 / 51 |",
    "| Rows dropped or original field values changed | 0 |",
    "| Export/reload comparison | Passed |",
    "",
    "Review flags were kept separate from structural failures: "
    + "; ".join(f"{label.lower()}: {count:,}" for label, count in review_counts.items())
    + ". These labels do not by themselves establish an error; no flagged record was removed.",
    "",
    "Four derived variables were added: decade, name length, share of published "
    "counts within state–sex–year, and descending count rank with alphabetical "
    "tie-breaking. Each group’s shares sum to one. Missing name records were not "
    "filled with zeros, spellings were preserved, and the CSV export was reloaded "
    "and checked against the prepared table.",
    "",
    "The original acquisition date and exact release remain undocumented. "
    "SHA-256 checks identify the local input snapshot; they do not authenticate "
    "its original download or resolve SSA coverage limitations.",
]
report_file = ROOT / 'reports/_preparation_results.md'
report_file.write_text(chr(10).join(report_lines) + chr(10), encoding='utf-8')
print(f'Wrote verified preparation summary: {report_file.relative_to(ROOT)}')

Wrote verified preparation summary: reports/_preparation_results.md
